# 04 — Comparison (Expedia, date split, test window) and CSV export

Reads the ranks written by notebooks 01, 02 and 03 (`data/interim/week6/`) and writes the result tables of `results/week6_retrain/`. **Primary protocol: autoregressive (AR)**, NDCG@K first; teacher-forced (TF) is reported next to it, never mixed. Mean ± sample std over the seeds (ItemKNN is deterministic). Paired bootstrap 95 % intervals resample **users** (AR rows of one user are dependent), reference seed. Also writes the train / valid / test CSVs of the date split (column `hotel_cluster`); the test CSV carries the main model's AR top-10 per row.

Limitations: `hotel_cluster` is a proxy for room category / package; one dataset; offline only; two simple baselines; no retraining on train + valid; valid (Sep-Oct) and test (Nov-Dec) cover different months; AR measures planning without feedback, TF is the closer proxy of a deployed system.

In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

import numpy as np
import torch  # torch before pandas-importing modules (Windows c10.dll)
import yaml

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

from nbp.data.split import TRAIN
from nbp.eval import runner as R
from nbp.eval.report import metrics, row_set_masks, write_json
from nbp.paths import CONFIGS, EXPERIMENTS, INTERIM, PROCESSED

SPLIT = yaml.safe_load((CONFIGS / "split.yaml").read_text(encoding="utf-8"))
TRAIN_CFG = yaml.safe_load((CONFIGS / "train.yaml").read_text(encoding="utf-8"))
MODEL_CFG = yaml.safe_load((CONFIGS / "smlprec_expedia.yaml").read_text(encoding="utf-8"))
assert MODEL_CFG["MAX_ITEM_LIST_LENGTH"] == SPLIT["max_history"], "history length differs between configs"
SEEDS = list(SPLIT["seeds"])
REF_SEED = SEEDS[0]  # grids and sameDest weights are chosen on this seed
FAST = os.environ.get("NBP_FAST") == "1"  # smoke run: a user subsample, short training, outputs under *_fast
FAST_FRAC = float(os.environ.get("NBP_FAST_FRAC", "0.02"))
if FAST:
    TRAIN_CFG["max_epochs"] = int(os.environ.get("NBP_FAST_EPOCHS", "2"))
RES = ROOT / "results" / ("week6_retrain_fast" if FAST else "week6_retrain")
RANKS = INTERIM / ("week6_fast" if FAST else "week6")
EXP = EXPERIMENTS / "_fast" if FAST else EXPERIMENTS
TODAY = f"{date.today():%Y-%m-%d}"
K_MON = int(TRAIN_CFG["monitor"].split("@")[1])
BATCH = int(TRAIN_CFG["eval_batch"])
torch.set_num_threads(int(os.environ.get("NBP_THREADS", torch.get_num_threads())))

rows_full = R.load_rows(PROCESSED / "expedia_date_split" / "rows.npz")
codes = rows_full["codes"]  # (N, F) query codes, row = global event index
N_ITEMS = int(rows_full["y"].max()) + 1  # model ids 0 (padding) .. number of clusters
rows = R.subsample_users(rows_full, FAST_FRAC, 0) if FAST else rows_full
views = R.window_views(rows)
idx = R.window_index(rows)
Y = {w: rows["y"][i].astype(np.int64) for w, i in idx.items()}
L = {w: rows["L"][i] for w, i in idx.items()}
H = {w: rows["h"][i] for w, i in idx.items()}
USERS = {w: rows["user"][i] for w, i in idx.items()}
probe, y_probe = R.train_probe_view(rows, TRAIN_CFG["train_probe_rows"], TRAIN_CFG["train_probe_seed"])
print("FAST" if FAST else "FULL", {w: len(v) for w, v in views.items()}, "| items", N_ITEMS, "| threads", torch.get_num_threads())

FULL {'valid': 393399, 'test': 378788} | items 101 | threads 12


In [2]:
import pandas as pd

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import KS
from nbp.eval.report import H_BUCKETS, h_slices, metric_columns, protocol_gap

MODELS = {  # key -> (table name, seeds)
    "itemknn": ("ItemKNN", [None]),
    "plain": ("SMLP4Rec plain", SEEDS),
    "logreg": ("Logistic regression", SEEDS),
    "query": ("SMLP4Rec + query embedding", SEEDS),
    "main": ("SMLP4Rec + query embedding + sameDest (main)", SEEDS),
}
TRAINED_ON = {"itemknn": "train window, all bookings", "plain": "train window, L >= 1", "logreg": "train window, all bookings",
              "query": "train window, all bookings", "main": "train window, all bookings"}


def load(key, seed):
    name = f"ranks_{key}" + ("" if seed is None else f"_s{seed}") + ".npz"
    z = np.load(RANKS / name)
    for w in ("valid", "test"):
        assert np.array_equal(z[f"event_{w}"], rows["event"][idx[w]]), f"{name}: rows differ from the split"
    return {k: z[k] for k in z.files}


RK = {(m, s): load(m, s) for m, (_, seeds) in MODELS.items() for s in seeds}
masks = row_set_masks(L["test"])
long = []
for (m, s), r in RK.items():
    for proto in ("ar", "tf"):
        for rs, mk in masks.items():
            long.append({"dataset": "expedia", "model": MODELS[m][0], "trained_on": TRAINED_ON[m], "seed": "deterministic" if s is None else s,
                         "window": "test", "row_set": rs, "protocol": proto, **metrics(r[f"test_{proto}"], mk)})
bench = pd.DataFrame(long)
RES.mkdir(parents=True, exist_ok=True)
bench.round(4).to_csv(RES / "final_benchmark.csv", index=False)
print(len(bench), "rows ->", RES / "final_benchmark.csv")

78 rows -> C:\workspace\PROJECTS\next-best-product-recommendation\results\week6_retrain\final_benchmark.csv


In [3]:
COLS = metric_columns(KS)
agg = bench.groupby(["protocol", "row_set", "model"], sort=False)
mean, std, n_seeds = agg[COLS].mean(), agg[COLS].std(ddof=1), agg.size()
order = [MODELS[m][0] for m in MODELS]


def fmt_table(proto, rs):
    lines = [f"### Test, row set = {rs}, protocol = {'autoregressive (primary)' if proto == 'ar' else 'teacher-forced (secondary)'} (Expedia, date split)", "",
             "| model | seeds | n | " + " | ".join(COLS) + " |", "|---|---|---|" + "---|" * len(COLS)]
    for mname in order:
        key = (proto, rs, mname)
        ns = int(n_seeds[key])
        n = int(bench[(bench.protocol == proto) & (bench.row_set == rs) & (bench.model == mname)]["n"].iloc[0])
        cells = [f"{mean.loc[key, c]:.4f}" + (f" ± {std.loc[key, c]:.4f}" if ns > 1 else "") for c in COLS]
        lines.append(f"| {mname} | {ns} | {n} | " + " | ".join(cells) + " |")
    return "\n".join(lines)


md = ["<!-- generated by notebooks/hospitality/smlp4rec/04_compare.ipynb; do not edit -->", "",
      "Mean ± sample std over seeds; ItemKNN is deterministic. Cold rows: the plain model scores an empty history (one constant ranking); ItemKNN uses first-booking popularity.", ""]
for proto in ("ar", "tf"):
    for rs in ("all", "warm", "cold"):
        md += [fmt_table(proto, rs), ""]

## Bootstrap (reference seed, users resampled), AR minus TF gaps, h slices, check-in month slices

In [4]:
B = int(TRAIN_CFG["bootstrap_resamples"])
users = USERS["test"]
PAIRS = [("main", "logreg"), ("main", "itemknn"), ("main", "query"), ("query", "plain"), ("plain", "itemknn")]


def rk(m, proto):
    return RK[(m, None if m == "itemknn" else REF_SEED)][f"test_{proto}"]


boot = []
for a, b in PAIRS:
    for proto in ("ar", "tf"):
        for rs, mk in masks.items():
            for k in KS:
                res_ = paired_bootstrap(rk(a, proto)[mk], rk(b, proto)[mk], k=k, n_boot=B, seed=0, groups=users[mk])
                for met, v in res_.items():
                    boot.append({"protocol": proto, "row_set": rs, "comparison": f"{MODELS[a][0]} minus {MODELS[b][0]}", "metric": met,
                                 "diff": v["diff"], "ci95_low": v["ci95"][0], "ci95_high": v["ci95"][1], "n": int(mk.sum())})
boot = pd.DataFrame(boot)
boot.to_csv(RES / "bootstrap_ci.csv", index=False)

gap = []
for m in MODELS:
    for rs, mk in masks.items():
        g = protocol_gap(rk(m, "ar")[mk], rk(m, "tf")[mk], users[mk], KS, B, 0)
        for met, v in g.items():
            gap.append({"model": MODELS[m][0], "row_set": rs, "metric": met, "ar_minus_tf": v["diff"], "ci95_low": v["ci95"][0], "ci95_high": v["ci95"][1]})
gap = pd.DataFrame(gap)
gap.to_csv(RES / "protocol_gap.csv", index=False)

hs = []
for m in MODELS:
    for proto in ("ar", "tf"):
        for b_, v in h_slices(rk(m, proto), H["test"]).items():
            hs.append({"model": MODELS[m][0], "protocol": proto, "h_bucket": b_, **v})
hs = pd.DataFrame(hs)
hs.round(4).to_csv(RES / "slices_h.csv", index=False)

month = rows["checkin_month"][idx["test"]]
ms = []
for m in MODELS:
    for mo in np.unique(month):
        ms.append({"model": MODELS[m][0], "protocol": "ar", "checkin_month": int(mo), **metrics(rk(m, "ar"), month == mo)})
pd.DataFrame(ms).round(4).to_csv(RES / "slices_checkin_month.csv", index=False)

md += ["### Paired bootstrap 95 % CI (users resampled), reference seed, test, row set = all, NDCG@10 / Recall@10", "",
       "| protocol | comparison | metric | diff | ci95 |", "|---|---|---|---|---|"]
for _, r_ in boot[(boot.row_set == "all") & boot.metric.isin(["ndcg@10", "recall@10"])].iterrows():
    md.append(f"| {r_.protocol} | {r_.comparison} | {r_.metric} | {r_['diff']:+.4f} | [{r_.ci95_low:+.4f}, {r_.ci95_high:+.4f}] |")
md += ["", "### AR minus TF, reference seed, test, row set = all", "", "| model | metric | AR - TF | ci95 |", "|---|---|---|---|"]
for _, r_ in gap[(gap.row_set == "all") & gap.metric.isin(["ndcg@10", "recall@10"])].iterrows():
    md.append(f"| {r_.model} | {r_.metric} | {r_.ar_minus_tf:+.4f} | [{r_.ci95_low:+.4f}, {r_.ci95_high:+.4f}] |")
md += ["", "### NDCG@10 by h (earlier same-window bookings of the user), reference seed, test", "",
       "| model | protocol | " + " | ".join(n for _, _, n in H_BUCKETS) + " |", "|---|---|" + "---|" * len(H_BUCKETS)]
for (mname, proto), grp in hs.groupby(["model", "protocol"], sort=False):
    md.append(f"| {mname} | {proto} | " + " | ".join(f"{grp.set_index('h_bucket').loc[n, 'ndcg@10']:.4f}" for _, _, n in H_BUCKETS) + " |")
md += ["", "h-bucket row counts: " + ", ".join(f"{n} = {int(hs[(hs.model == MODELS['main'][0]) & (hs.protocol == 'ar') & (hs.h_bucket == n)]['n'].iloc[0])}" for _, _, n in H_BUCKETS)]
(RES / "final_tables.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("\n".join(md))

<!-- generated by notebooks/hospitality/smlp4rec/04_compare.ipynb; do not edit -->

Mean ± sample std over seeds; ItemKNN is deterministic. Cold rows: the plain model scores an empty history (one constant ranking); ItemKNN uses first-booking popularity.

### Test, row set = all, protocol = autoregressive (primary) (Expedia, date split)

| model | seeds | n | ndcg@5 | ndcg@10 | ndcg@20 | recall@5 | recall@10 | recall@20 |
|---|---|---|---|---|---|---|---|---|
| ItemKNN | 1 | 378788 | 0.1445 | 0.1825 | 0.2248 | 0.2169 | 0.3354 | 0.5036 |
| SMLP4Rec plain | 3 | 378788 | 0.1463 ± 0.0003 | 0.1844 ± 0.0005 | 0.2267 ± 0.0004 | 0.2195 ± 0.0007 | 0.3381 ± 0.0009 | 0.5064 ± 0.0013 |
| Logistic regression | 3 | 378788 | 0.3246 ± 0.0065 | 0.3872 ± 0.0063 | 0.4315 ± 0.0057 | 0.4851 ± 0.0081 | 0.6787 ± 0.0074 | 0.8529 ± 0.0048 |
| SMLP4Rec + query embedding | 3 | 378788 | 0.3817 ± 0.0029 | 0.4414 ± 0.0029 | 0.4804 ± 0.0024 | 0.5503 ± 0.0038 | 0.7345 ± 0.0036 | 0.8878 ± 0.0018 |
| SMLP4Rec + query em

## CSV export of the date split (column `hotel_cluster`); test CSV with the main model's AR top-10

In [5]:
from nbp.data.split import WINDOW_NAMES

bk = pd.read_parquet(INTERIM / "expedia_bookings.parquet")
assert len(bk) == len(rows_full["y"]) and (bk["item_id"].to_numpy() + 1 == rows_full["y"]).all()
keep = [c for c in bk.columns if not c.startswith("tgt_") and c not in ("aux_cnt", "event_type", "domain", "dataset")]
base = bk[keep].rename(columns={"item_id": "hotel_cluster"})
base.insert(0, "event_id", np.arange(len(bk)))
base["n_prior_bookings"] = rows_full["L"]
base["n_prior_same_window"] = rows_full["h"]
base["is_cold"] = rows_full["L"] == 0
TOPK = int(TRAIN_CFG["top_k_export"])
main_ref = RK[("main", REF_SEED)]
top = main_ref["test_ar_top"][:, :TOPK].astype(np.int64) - 1  # model id -> hotel_cluster
assert (top >= 0).all()
CSV = PROCESSED / ("expedia_date_split_fast" if FAST else "expedia_date_split") / "csv"
CSV.mkdir(parents=True, exist_ok=True)
counts = {}
for w, name in WINDOW_NAMES.items():
    part = base[rows_full["window"] == w].copy()
    if name == "test":
        ev_test = rows["event"][idx["test"]]
        part = part[np.isin(part["event_id"].to_numpy(), ev_test)]  # whole window (FAST: the user subsample)
        assert np.array_equal(part["event_id"].to_numpy(), ev_test)
        part[f"top{TOPK}_main_ar"] = [" ".join(map(str, t)) for t in top]
        part["truth_rank_main_ar"] = main_ref["test_ar"]
    part.to_csv(CSV / f"{name}.csv", index=False)
    counts[name] = int(len(part))
hit = main_ref["test_ar"] <= TOPK
write_json(RES / "csv_export.json", {"folder": str(CSV.relative_to(ROOT)), "rows": counts, "top_k": TOPK,
           "model": MODELS["main"][0], "seed": REF_SEED, "protocol": "autoregressive",
           f"test_recall@{TOPK}_from_csv_column": float(hit.mean()), "columns_test": list(part.columns)})
print(counts, "| test recall@10 of the exported top-10:", round(float(hit.mean()), 4))

{'train': 2180488, 'valid': 393399, 'test': 378788} | test recall@10 of the exported top-10: 0.741
